In [1]:
"""
run_all_0413.py — Master experiment runner for EXP/0413.

Grid:
  Products (3):  Sparkling water, Cola, Protein bar
  Suppliers (2): LOCAL (S1), NATIONAL (S2)
  Buyer types (8): random, wild,
                   persistent@5_levels,
                   concession, anchor_drag
  Seller strategies (6): cooperative, anchor_high, gradual_conceder,
                          tit_for_tat_matcher, fairness_responder,
                          walk_away_shutdown

Fixed: model=deepseek-reasoner, n_rounds=12, n_runs=5

Output layout:
  results/<seller_strategy>/<buyer_type>__<product_short>__<supplier_id>.json
  results/<seller_strategy>/<buyer_type>__all_transitions.json

Usage:
  # Run everything:
    python run_all_0413.py

  # Run one seller strategy:
    python run_all_0413.py cooperative

  # Run one seller strategy + one buyer type:
    python run_all_0413.py cooperative random
"""

import sys
from pathlib import Path

from openai import OpenAI

from NegoLib.Entities.Agents import LOCAL_SUPPLIER, NATIONAL_SUPPLIER
from NegoLib.Entities.products import COLA, PROTEIN_BAR, EARPHONES

from exp_runner_0413 import (
    RandomBuyer, WildBuyer, PersistentBuyer, BudgetBuyer,
    ConcessionBuyer, AnchorDragBuyer,
    run_condition, build_transitions, save_json,
)

# ═══════════════════════════════════════════════════════════════
# CONFIG — edit these
# ═══════════════════════════════════════════════════════════════
CLIENT = OpenAI(
    api_key="sk-81e3e6eae33d4486ad4127e0cb3630a6",         # <-- replace
    base_url="https://api.deepseek.com",
)
MODEL = "deepseek-reasoner"
N_RUNS = 5
N_ROUNDS = 12
BASE_SEED = 5000

OUTPUT_ROOT = Path("results")

# ═══════════════════════════════════════════════════════════════
# Product × Supplier grid
# ═══════════════════════════════════════════════════════════════
PRODUCTS = [
    (EARPHONES, 24),
    (COLA,            60),
    (PROTEIN_BAR,     40),
]

SUPPLIERS = [LOCAL_SUPPLIER, NATIONAL_SUPPLIER]

PRODUCT_SHORT = {
    "Cola (330ml)":            "cola",
    "Protein bar (60g)":       "protein",
    "Earphones (basic)":       "earphones",
}

# ═══════════════════════════════════════════════════════════════
# Buyer types
# ═══════════════════════════════════════════════════════════════
def _build_buyers():
    buyers = {}

    # 1. Random buyer: Uniform(cost, market)
    buyers["random"] = RandomBuyer()

    # 2. Wild buyer: Uniform(0, 2*market)
    buyers["budget"] = BudgetBuyer()

    # 3. Persistent buyers at 5 price levels
    #    Levels are fractions of market_price: 0.2, 0.4, 0.6, 0.8, 1.2
    #    Actual price = frac * market_price, computed per-product in the loop
    #    We use a lambda-like approach: store frac, compute price at run time
    #    → handled specially in the run loop below
    # for frac in [0.1, 0.2,0.4, 0.6, 0.8, 1.0, 1.2]:
    #     buyers[f"persistent_frac{frac:.1f}"] = frac  # sentinel: resolved per-product

    # 4. Concession buyer: starts low, concedes upward
    buyers["concession"] = ConcessionBuyer(start_frac=0.1, end_frac=0.6)

    # 5. Anchor-drag buyer: low anchor, tiny steps
    buyers["anchor_drag"] = AnchorDragBuyer(anchor_frac=0.05, step_frac=0.03)

    return buyers


BUYER_SPECS = _build_buyers()

# ═══════════════════════════════════════════════════════════════
# Seller strategies
# ═══════════════════════════════════════════════════════════════
ALL_STRATEGIES = [
    "cooperative",
    "anchor_high",
    "gradual_conceder",
    "tit_for_tat_matcher",
    "fairness_responder",
    "walk_away_shutdown",
]


# ═══════════════════════════════════════════════════════════════
# Main loop
# ═══════════════════════════════════════════════════════════════
def run_one_strategy(strategy: str, buyer_filter: str | None = None):
    """Run all buyer × product × supplier conditions for one seller strategy."""
    strat_dir = OUTPUT_ROOT / strategy
    strat_dir.mkdir(parents=True, exist_ok=True)

    for buyer_name, buyer_spec in BUYER_SPECS.items():
        if buyer_filter and buyer_name != buyer_filter:
            continue

        all_transitions_for_buyer = []

        for product, qty in PRODUCTS:
            pshort = PRODUCT_SHORT.get(product.name, product.name[:8])

            for supplier in SUPPLIERS:
                # Resolve persistent buyers to actual price
                if isinstance(buyer_spec, float):
                    # buyer_spec is a fraction of market_price
                    actual_price = round(buyer_spec * product.market_price, 2)
                    buyer = PersistentBuyer(fixed_price=actual_price)
                else:
                    buyer = buyer_spec

                v = product.cost + supplier.internal_costs.for_category(product.category)
                tag = f"{buyer_name}__{pshort}__{supplier.id}"
                print(f"\n{'='*60}")
                print(f"  Strategy={strategy}  Buyer={buyer_name}  "
                      f"Product={pshort}  Supplier={supplier.id}  v={v:.2f}")
                print(f"{'='*60}")


                if buyer_name.startswith("persistent") and isinstance(buyer_spec, float):
                    n_runs_specify = 1
                elif buyer_name == "wild":
                    n_runs_specify = N_RUNS * 3
                else:
                    n_runs_specify = N_RUNS * 2

                experiment = run_condition(
                    client=CLIENT, model=MODEL,
                    product=product, quantity=qty,
                    supplier=supplier, strategy=strategy,
                    buyer=buyer,
                    n_runs=n_runs_specify, n_rounds=N_ROUNDS,
                    base_seed=BASE_SEED,
                    verbose=True,
                )

                # Save per-condition file
                save_json(experiment, strat_dir / f"{tag}.json")

                # Collect transitions
                trans = build_transitions(experiment)
                all_transitions_for_buyer.extend(trans)

        # Save merged transitions for this buyer type
        if all_transitions_for_buyer:
            save_json(
                {"strategy": strategy, "buyer_type": buyer_name,
                 "n_transitions": len(all_transitions_for_buyer),
                 "transitions": all_transitions_for_buyer},
                strat_dir / f"{buyer_name}__all_transitions.json",
            )
            print(f"\n  [{strategy}/{buyer_name}] Total transitions: {len(all_transitions_for_buyer)}")


# Anchor high

In [ ]:
strategy = ALL_STRATEGIES[1]
buyer_filter = None
print(f"  SELLER STRATEGY: {strategy}")
print(f"{'#'*70}")
run_one_strategy(strategy, buyer_filter)

# 1165 min

  SELLER STRATEGY: anchor_high
######################################################################

  Strategy=anchor_high  Buyer=random  Product=earphones  Supplier=S1  v=4.60

  --- Run 1/10 (seed=5000) ---
  R 1 BUY $3.25 T6  SEL $6.50  "Thank you for your interest. However, $3.25 per unit is significantly below our ..."
  R 2 BUY $4.62 T6  SEL $6.50  "I appreciate your counter, but $4.62 is still not a sustainable price for us con..."
  R 3 BUY $7.00 T5  SEL $7.25  "I appreciate your increased offer. While $7.00 is closer to the product's value,..."
  R 4 BUY $4.26 T4  SEL $7.15  "I understand you're looking for the best possible price, but $4.26 is simply not..."
  R 5 BUY $5.46 T1  SEL $6.75  "Thank you for moving closer to a reasonable price. However, $5.46 per unit still..."
  R 6 BUY $7.76 T6  SEL $7.90  "I appreciate your latest offer. At $7.76, you are recognizing the value and qual..."
  R 7 BUY $5.42 T0  SEL $7.50  "Thank you for your continued interest. While $5.42 rep